# 📊 Sesión 02 — Manipulación, visualización y pipeline integrador (R)
## 🪞 Notebook espejo R

**Curso:** Fundamentos de programación en Python y R · **PEBIBA XVIII** · 2026
**Docente:** Cand. Dr. Jordan King Rodríguez Mallqui · UNI

> Espejo del notebook Python. Misma narrativa, sintaxis tidyverse.

---

## 📑 Índice

1. Recapitulación día 1
2. Filter y select con dplyr
3. group_by + summarise
4. Pipe `|>` — encadenar operaciones
5. ⚠️ Trampa #4: Vectorización implícita y reciclaje
6. Visualización con ggplot2
7. Heurística Python vs R
8. 🎯 Mini-pipeline integrador
9. 🏆 Resumen y comparación con Python

## 💼 Caso de Negocio (continuación)

Mismo dataset Acme Corp. Hoy respondes a tu jefe en R **y** decides si tu trabajo del lunes lo harás en Python o R.

## 📦 Imports y configuración

In [ ]:
library(tidyverse)

options(scipen = 999)
options(tibble.print_max = 10)
options(repr.plot.width = 10, repr.plot.height = 6)  # tamaño default de plots en Jupyter

DATASET_FILENAME <- 'acme_ventas.csv'
DATA_DIR <- '../../01_Sesion_Sintaxis_Carga/data'

# Carga + columna calculada
df <- read_csv(file.path(DATA_DIR, DATASET_FILENAME)) |>
  mutate(ingresos = cantidad * precio_unit * (1 - descuento_pct))

cat('Dataset:', nrow(df), 'filas x', ncol(df), 'columnas\n')
cat('Ingresos rango:', round(min(df$ingresos), 0), 'a', round(max(df$ingresos), 0), 'PEN\n')

## 1. Recapitulación día 1

Verifica que conservas conceptos clave.

In [ ]:
# Q1: ¿primer elemento de c(10, 20, 30)?
vec <- c(10, 20, 30)
cat('Q1: vec[1] =', vec[1], '\n')   # ← R empieza en 1, no 0

# Q2: ¿qué pasa con copy-on-modify?
x <- c(1, 2, 3)
y <- x
y <- c(y, 4)
cat('Q2: x =', x, '   (intacto, R copy-on-modify)\n')

# Q3: ¿filas duplicadas?
cat('Q3: duplicadas =', sum(duplicated(df)), '\n')

## 2. Filter y select con dplyr

**Idea clave:** `filter()` para FILAS, `select()` para COLUMNAS.

In [ ]:
# Filter — filas con quarter == Q4
ventas_q4 <- df |> filter(quarter == 'Q4')
cat('Filas en Q4:', nrow(ventas_q4), '\n')

# Filter combinado
ventas_q4_lima <- df |> filter(quarter == 'Q4', region == 'Lima')
cat('Filas Q4 Lima:', nrow(ventas_q4_lima), '\n')

# Select columnas
df |> select(region, producto, ingresos) |> head(5)

### 💡 Pro-Tip: filter con múltiples condiciones

En `filter(...)` separar condiciones con coma equivale a AND:
```r
filter(quarter == 'Q4', region == 'Lima')  # AND implícito
filter(quarter == 'Q4' | region == 'Lima') # OR explícito
```

**Comparación con Python:**
- pandas: usa `&` y `|` con paréntesis: `df.loc[(df.q == 'Q4') & (df.region == 'Lima')]`.
- dplyr: comas implícitas para AND, `|` para OR. Más legible.

### 🧠 Micro-Desafío 2.1

Filtra ventas en Cusco con descuento ≥ 0.10. ¿Cuántas son?

In [ ]:
# Solución
ventas_cusco_descuento <- df |> filter(region == 'Cusco', descuento_pct >= 0.10)
cat('Ventas Cusco descuento >= 10%:', nrow(ventas_cusco_descuento), '\n')

## 3. group_by + summarise

In [ ]:
# Pregunta 1: ingresos totales por región
ingresos_region <- df |>
  group_by(region) |>
  summarise(ingresos_totales = sum(ingresos)) |>
  arrange(desc(ingresos_totales))

ingresos_region

In [ ]:
# Pregunta 2: agregaciones múltiples
resumen_region <- df |>
  group_by(region) |>
  summarise(
    n_transacciones = n(),
    ingresos_totales = sum(ingresos),
    ingreso_promedio = mean(ingresos),
    cantidad_total = sum(cantidad),
    .groups = 'drop'
  ) |>
  arrange(desc(ingresos_totales))

resumen_region

### ⚠️ Real-World Warning: `.groups = 'drop'`

Después de `group_by()`, dplyr deja el resultado **agrupado** (próximas operaciones aplican por grupo). Esto sorprende a alumnos nuevos.

Solución: pasar `.groups = 'drop'` al final del `summarise()`. Tiene equivalencia con `.reset_index()` de pandas.

In [ ]:
# Pregunta 3: agrupar por DOS columnas — pivot wider
matriz_region_quarter <- df |>
  group_by(region, quarter) |>
  summarise(ingresos = sum(ingresos), .groups = 'drop') |>
  pivot_wider(names_from = quarter, values_from = ingresos)

matriz_region_quarter

## 4. Pipe `|>` — encadenar operaciones

**Idea clave:** el pipe nativo `|>` (R 4.1+) lee de izquierda a derecha como una receta.

**Equivalente a method chaining de Python.** Más legible que asignar variables intermedias.

In [ ]:
# Top 5 productos por ingresos en Q4
top5_q4 <- df |>
  filter(quarter == 'Q4') |>
  group_by(producto) |>
  summarise(ingresos = sum(ingresos), .groups = 'drop') |>
  arrange(desc(ingresos)) |>
  slice_head(n = 5)

top5_q4

### 🪞 Comparación con Python (method chaining)

| dplyr (R) | pandas (Python) |
|---|---|
| `filter(quarter == 'Q4')` | `.query("quarter == 'Q4'")` |
| `group_by(producto)` | `.groupby('producto')` |
| `summarise(x = sum(...))` | `.agg(x=('col', 'sum'))` |
| `arrange(desc(x))` | `.sort_values('x', ascending=False)` |
| `slice_head(n = 5)` | `.head(5)` |
| `\|>` | method chaining |

### 🧠 Micro-Desafío 4.1

Reescribe en una sola cadena: 3 productos con MENOR ingreso promedio en Lima durante Q1.

In [ ]:
# Solución
bottom3 <- df |>
  filter(region == 'Lima', quarter == 'Q1') |>
  group_by(producto) |>
  summarise(ingreso_prom = mean(ingresos), .groups = 'drop') |>
  arrange(ingreso_prom) |>
  slice_head(n = 3)

bottom3

## 5. ⚠️ Trampa #4: Vectorización y reciclaje

**En R todo es vector.** Las operaciones aritméticas son automáticamente elemento-a-elemento.

**Pero ¡ojo!** R recicla silenciosamente vectores de tamaños distintos.

In [ ]:
# Vectorización elemento-a-elemento — natural en R
x <- c(1, 2, 3, 4, 5, 6)
y <- c(10, 20, 30, 40, 50, 60)
cat('x * y =', x * y, '\n')   # [10, 40, 90, 160, 250, 360]

In [ ]:
# ⚠️ TRAMPA — tamaños distintos: R recicla silenciosamente
x <- c(1, 2, 3, 4, 5, 6)
y <- c(10, 20)             # solo 2 elementos
cat('x * y =', x * y, '\n')
# Resultado: [10, 40, 30, 80, 50, 120] — y se recicló como [10,20,10,20,10,20]

In [ ]:
# ⚠️ Caso peor — tamaños no múltiplos: R AVISA pero igual ejecuta
x <- c(1, 2, 3, 4, 5)   # 5 elementos
y <- c(10, 20)           # 2 elementos (5 no es múltiplo de 2)
tryCatch(
  print(x * y),
  warning = function(w) cat('R warning:', conditionMessage(w), '\n')
)

### ⚠️ Real-World Warning

Si tu cálculo combina vectores que vienen de fuentes distintas, **siempre verifica `length()`** antes de operar. R puede dar resultados "limpios" pero numéricamente incorrectos.

## 6. Visualización con ggplot2

**Filosofía:** ggplot2 es **declarativo** — describes capas y agregaciones con `+`.

Los 4 gráficos clave del trabajo:

In [ ]:
# 1) Barras: ingresos por región
ggplot(ingresos_region, aes(x = reorder(region, -ingresos_totales), y = ingresos_totales)) +
  geom_col(fill = '#2C3E50') +
  labs(
    title = 'Ingresos totales por región — Acme Corp 2026',
    x = 'Región',
    y = 'Ingresos (PEN)'
  ) +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = 'bold'))

In [ ]:
# 2) Línea: tendencia trimestral
df |>
  group_by(quarter) |>
  summarise(ingresos = sum(ingresos), .groups = 'drop') |>
  ggplot(aes(x = quarter, y = ingresos, group = 1)) +
    geom_line(color = '#8A1525', linewidth = 1.5) +
    geom_point(color = '#8A1525', size = 3) +
    labs(title = 'Evolución trimestral de ingresos',
         x = 'Trimestre', y = 'Ingresos (PEN)') +
    theme_minimal(base_size = 12) +
    theme(plot.title = element_text(face = 'bold'))

In [ ]:
# 3) Histograma: distribución de descuentos
ggplot(df, aes(x = descuento_pct)) +
  geom_histogram(bins = 20, fill = '#10B981', color = 'white') +
  labs(title = 'Distribución de descuentos aplicados',
       x = 'Descuento %', y = 'Frecuencia') +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = 'bold'))

In [ ]:
# 4) Heatmap: ingresos por región × quarter
df |>
  group_by(region, quarter) |>
  summarise(ingresos = sum(ingresos)/1000, .groups = 'drop') |>
  ggplot(aes(x = quarter, y = region, fill = ingresos)) +
    geom_tile() +
    geom_text(aes(label = round(ingresos, 0)), color = 'white') +
    scale_fill_gradient(low = '#EF4444', high = '#10B981') +
    labs(title = 'Mapa de ingresos por región × trimestre',
         x = 'Trimestre', y = 'Región', fill = 'Miles PEN') +
    theme_minimal(base_size = 12) +
    theme(plot.title = element_text(face = 'bold'))

### 🪞 Comparación visual: matplotlib vs ggplot2

| Aspecto | matplotlib (Python) | ggplot2 (R) |
|---|---|---|
| Filosofía | Imperativa: agregar elementos | Declarativa: describir capas |
| Sintaxis | `plt.bar(...)`, `plt.title(...)` | `ggplot(df, aes(...)) + geom_X() + labs(...)` |
| Curva | Más verbosa | Una vez aprendida, escala mejor |
| Calidad publicación | Buena con esfuerzo | Excelente por defecto |

## 7. Heurística Python vs R

Misma tabla del notebook Python — está en `S02_Manipulacion_y_Pipeline_PY.ipynb` sección 7.

**Productive Failure:** decide tus 5 escenarios ANTES de la discusión grupal.

## 8. 🎯 Mini-pipeline integrador (R)

In [ ]:
# Función pipeline reproducible
construir_pipeline_top_productos <- function(df, quarter_objetivo = 'Q4', top_n = 5) {
  df |>
    filter(quarter == quarter_objetivo) |>
    filter(cantidad < 1000) |>
    group_by(region, producto) |>
    summarise(
      ingresos_totales = sum(ingresos),
      unidades_vendidas = sum(cantidad),
      .groups = 'drop'
    ) |>
    group_by(region) |>
    mutate(rank = dense_rank(desc(ingresos_totales))) |>
    filter(rank <= top_n) |>
    arrange(region, rank) |>
    ungroup()
}

# Ejecutar
top_productos <- construir_pipeline_top_productos(df, 'Q4', 5)
cat('Resultado:', nrow(top_productos), 'filas\n')
head(top_productos, 10)

In [ ]:
# Asserts auto-verificables
stopifnot(nrow(top_productos) <= 30)
stopifnot(min(top_productos$ingresos_totales) > 0)
stopifnot(max(top_productos$rank) <= 5)
cat('OK Pipeline R pasa todos los asserts\n')

In [ ]:
# Visualización del resultado
top_productos |>
  mutate(label = paste(region, producto, sep = ' - ')) |>
  ggplot(aes(x = ingresos_totales, y = reorder(label, ingresos_totales), fill = region)) +
    geom_col() +
    labs(title = 'Top 5 productos por ingresos Q4 — por región',
         x = 'Ingresos (PEN)', y = '') +
    theme_minimal(base_size = 11) +
    theme(plot.title = element_text(face = 'bold'),
          legend.position = 'none')

In [ ]:
# Exportar a CSV/Excel para Power BI
if (!dir.exists('../outputs')) dir.create('../outputs')
write_csv(top_productos, '../outputs/top5_productos_q4_por_region_R.csv')
cat('OK Exportado: top5_productos_q4_por_region_R.csv\n')

## 🏆 9. Resumen y comparación con Python

**Hoy aprendiste R completo:**

1. ✅ filter / select con dplyr
2. ✅ group_by + summarise + .groups='drop'
3. ✅ Pipe nativo `|>` para encadenar
4. ✅ Trampa #4: vectorización y reciclaje silencioso
5. ✅ Visualización con ggplot2 (declarativo)
6. ✅ Heurística para decidir Python vs R
7. ✅ Pipeline reproducible end-to-end en R

### Tabla resumen S02 — Python ↔ R

| Operación | pandas | dplyr |
|---|---|---|
| Filtrar | `.query()` o `.loc[]` | `filter()` |
| Seleccionar | `df[['a','b']]` | `select(a, b)` |
| Agrupar | `.groupby('col')` | `group_by(col)` |
| Agregar | `.agg(x=('y','sum'))` | `summarise(x = sum(y))` |
| Ordenar desc | `.sort_values('x', ascending=False)` | `arrange(desc(x))` |
| Top N | `.head(n)` | `slice_head(n=...)` |
| Crear columna | `.assign(z=...)` | `mutate(z = ...)` |
| Pivot ancho | `.pivot()` o `.unstack()` | `pivot_wider()` |
| Pivot largo | `.melt()` | `pivot_longer()` |
| Encadenar | method chaining `()` | pipe `\|>` |
| Tabla a Excel | `df.to_excel(path)` | `write_xlsx(path)` (paquete `writexl`) |

→ Esta tabla está expandida en `S02_Manipulacion_y_Pipeline_ROSETTA.md`.

---

**¡Felicidades! 8 horas y construiste un pipeline reproducible en DOS lenguajes.**

Mini-proyecto opcional en `07Evaluación/miniproyecto/enunciado.md`.

*Cierre Sesión 02 (espejo R) — Fundamentos de Python y R · PEBIBA XVIII · 2026*

---

## 🚀 Profundización R (opcional)

> **Solo si ya terminaste el mini-pipeline** y quieres explorar capacidades de R que pandas no replica con la misma elegancia.

Tres bloques cortos:

1. **ggplot2 con facet_wrap** — comparativa multi-panel automática
2. **dplyr::across** — aplicar funciones a múltiples columnas a la vez
3. **RMarkdown / Quarto** — reportes reproducibles (territorio donde R sigue siendo único)


### A. ggplot2 + facet_wrap — un panel por categoría

Cuando quieres comparar varias regiones en un solo gráfico, `facet_wrap` divide la figura en sub-paneles automáticamente. En matplotlib necesitas un loop con `subplot`.

In [ ]:
library(ggplot2)
library(dplyr)

# Top 5 productos por region — un panel por region
top5_por_region <- df |>
  mutate(ingreso = precio_unit * cantidad) |>
  group_by(region, producto) |>
  summarise(total = sum(ingreso), .groups = 'drop') |>
  group_by(region) |>
  slice_max(total, n = 5)

ggplot(top5_por_region, aes(x = reorder(producto, total), y = total, fill = region)) +
  geom_col() +
  coord_flip() +
  facet_wrap(~ region, scales = 'free_y') +
  labs(title    = 'Top 5 productos por region — Acme Corp',
       subtitle = 'Ingreso total (cantidad x precio unitario)',
       x = NULL, y = 'Ingreso total (PEN)',
       caption = 'Fuente: dataset sintetico · 2026') +
  theme_minimal(base_size = 11) +
  theme(legend.position = 'none',
        plot.title = element_text(face = 'bold', size = 14))

**Lo que acabas de ver:**

- `facet_wrap(~ region, scales = 'free_y')` → 6 paneles automáticos, uno por región. Cada uno con su propia escala.
- `theme_minimal(base_size = 11)` → tipografía y márgenes coherentes para informe ejecutivo.
- `labs(title, subtitle, caption)` → título estructurado tipo *The Economist*.

*Esto es lo que hace que ggplot2 siga siendo estándar para visualización publicable en revistas académicas.*

### B. dplyr::across — aplicar la misma operación a múltiples columnas

Cuando quieres aplicar `mean`, `sum` o cualquier función a 5 columnas distintas, NO escribas 5 líneas. Usa `across`.

In [ ]:
library(dplyr)

# Estadistica descriptiva para todas las columnas numericas en un solo comando
df |>
  summarise(across(c(cantidad, precio_unit, descuento_pct),
                   list(min = min, max = max, mean = mean, sd = sd),
                   .names = '{.col}_{.fn}'))

# Equivalente en pandas:
# df[['cantidad','precio_unit','descuento_pct']].agg(['min','max','mean','std'])
# pandas es mas conciso aqui. across brilla cuando hay logica condicional sobre las columnas.

### C. RMarkdown / Quarto — reportes reproducibles

Esto es **el territorio donde R sigue siendo único**. Un archivo `.Rmd` (o `.qmd`) combina:

- Texto narrativo (Markdown)
- Código R que se ejecuta
- Tablas y gráficos generados en vivo
- Salida final: HTML, PDF, Word, presentación, libro, sitio web

**Caso de uso real:** un informe ejecutivo que se regenera cada lunes con datos frescos. Sin copiar-pegar gráficos. Un comando.

```yaml
---
title: "Reporte semanal de ventas"
author: "Equipo Analytics"
date: "`r format(Sys.Date(), '%d %B %Y')`"
output:
  html_document:
    toc: true
    theme: cosmo
---
```

El equivalente en Python existe (Jupyter Book, Quarto que también soporta Python), pero el ecosistema R lleva 15 años más desarrollado. **Si tu trabajo incluye reportes recurrentes, R + RMarkdown vale la pena aprender.**

### Cierre de la profundización

| Tema | Cuándo lo vas a usar |
|---|---|
| `facet_wrap` y themes ggplot2 | Visualización publicable o ejecutiva |
| `dplyr::across` | Mismo cálculo sobre 3+ columnas |
| RMarkdown / Quarto | Reportes recurrentes que se regeneran solos |

*Para profundizar:* Wickham — **R for Data Science** (cap. visualización + comunicación) y la documentación de Quarto: https://quarto.org/